In [542]:
import random

In [543]:
initial_capital = 1000
historical_changes = [-1.2, 3.4, -0.8, 2.1, -2.5, 1.7, -0.3, 5.8, -1.1, 3.5]
mutation_rate = 0.05
generations = 10
population_size = 4

In [544]:
def encoder(chromosome):
    stop_loss = chromosome["stop_loss"]
    take_profit = chromosome["take_profit"]
    trade_size = chromosome["trade_size"]
    encoded_chromosome = ""

    if stop_loss < 10:
        encoded_chromosome += f"0{stop_loss}"
    else:
        encoded_chromosome += str(stop_loss)

    if take_profit < 10:
        encoded_chromosome += f"0{take_profit}"
    else:
        encoded_chromosome += str(take_profit)

    if trade_size < 10:
        encoded_chromosome += f"0{trade_size}"
    else:
        encoded_chromosome += str(trade_size)

    return encoded_chromosome

In [545]:
def generate_chromosome():
    chromosome = {}
    chromosome["stop_loss"] = random.randint(1, 99)
    chromosome["take_profit"] = random.randint(1, 99)
    chromosome["trade_size"] = random.randint(1, 99)
    return chromosome

In [546]:
def fitness(encoded_chromosome):
    stop_loss = int(encoded_chromosome[:2]) * (-1)
    take_profit = int(encoded_chromosome[2:4])
    trade_size = int(encoded_chromosome[4:])
    capital = initial_capital
    for i in range(len(historical_changes)):
        trade_amount = capital * (trade_size/100)
        if historical_changes[i] > take_profit:
            profit = trade_amount * (take_profit/100)
        elif historical_changes[i] < stop_loss:
            profit = trade_amount * (stop_loss/100)
        else:
            profit = trade_amount * (historical_changes[i]/100)
        capital += profit
    return capital - initial_capital

In [547]:
def crossover(p1, p2):
    point = random.randint(1, len(p1) - 1)
    off1 = p1[:point] + p2[point:]
    off2 = p2[:point] + p1[point:]
    return off1, off2

In [548]:
def mutate(encoded_chromosome):
    if random.random() < mutation_rate:
        index = random.randint(0, len(encoded_chromosome)-1)
        encoded_chromosome = encoded_chromosome[:index] + str(random.randint(0, 9)) + encoded_chromosome[index+1:]
    return encoded_chromosome

In [549]:
def run_ga():
    population = []
    for i in range(population_size):
        chromosome = generate_chromosome()
        encoded_chromosome = encoder(chromosome)
        population.append(encoded_chromosome)

    for gen in range(generations):
        population.sort(key=fitness, reverse=True)
        new_gen = population[:2]
        while len(new_gen) < population_size:
            p1, p2 = random.sample(population, 2)
            off1, off2 = crossover(p1, p2)
            new_gen.append(mutate(off1))
            if len(new_gen) < population_size:
                new_gen.append(mutate(off2))
        population = new_gen

    best_chromosome = population[0]
    stop_loss = int(best_chromosome[:2])
    take_profit = int(best_chromosome[2:4])
    trade_size = int(best_chromosome[4:])
    best = {}
    best["stop_loss"] = stop_loss
    best["take_profit"] = take_profit
    best["trade_size"] = trade_size
    return best, fitness(best_chromosome)

In [550]:
best, profit = run_ga()
print(f"Best Strategy: {best}")
print(f"Profit: {profit}")

Best Strategy: {'stop_loss': 13, 'take_profit': 13, 'trade_size': 98}
Profit: 105.60244523218694
